# Manufacturing Optimisation Platform — interactive demo

**No programming needed.** Click **Runtime → Run all** in the menu above, accept the
"not authored by Google" warning, and scroll down as results appear (first run takes ~1 minute
to install the solver).

Then play: change the sliders in the **"Try changing the world"** section and click the ▶
button on that cell to re-solve. You are re-running a real mixed-integer optimiser (HiGHS),
not a spreadsheet.

The case is **D02 — an Asahi-inspired east-coast DC network**: six candidate distribution-centre
sites, three state markets, five planning periods. Locations and the three-DC program are public
facts; all volumes and unit costs are illustrative estimates (see the provenance fields in the
data file). Full detail: [github.com/jkeetley1/manufacturing](https://github.com/jkeetley1/manufacturing).

In [ ]:
#@title 1. Set up (click Run all — this installs the solver and fetches the model) { display-mode: "form" }
!pip -q install "pyomo>=6.7" "highspy>=1.7,<1.16" pyyaml > /dev/null 2>&1
!rm -rf manufacturing && git clone -q --depth 1 https://github.com/jkeetley1/manufacturing.git
import sys; sys.path.insert(0, "manufacturing/golden/src")
from goldentest import load_problem, run_checks, solve
print("Ready.")

In [ ]:
#@title 2. Data gate — the Data Auditor checks the dataset before any solve { display-mode: "form" }
base = load_problem("manufacturing/demos/D02_asahi_inspired_dc_network.yaml")
dq = run_checks(base)
print(dq.summary())
for f in dq.findings:
    print(" ", f.severity.value, f.check, f.location, "-", f.message)
if dq.gate_passed and not dq.findings:
    print("No findings: units declared, geography consistent, provenance dated.")

In [ ]:
#@title 3. Solve the base case and draw the network { display-mode: "form" }
import copy, matplotlib.pyplot as plt

def show(problem, title):
    s = solve(problem)
    built = [i for i in s.open if any(s.open[i].values())]
    print(f"{title}")
    print(f"  NPV cost: {s.objective/1e9:.3f} bn USD   (optimality gap {s.reproducibility['optimisation_gap']:.0e})")
    for i in sorted(built):
        t0 = min(t for t in s.open[i] if s.open[i][t])
        print(f"  BUILD {i:15s} from {t0}, final capacity {s.expansion_units[i][problem.periods[-1]]*250} kt/yr")
    print(f"  not built: {sorted(set(s.open)-set(built))}")
    fig, ax = plt.subplots(figsize=(6,7))
    for c in problem.customers:
        ax.scatter(c.lon, c.lat, s=220, c="#2b6cb0", marker="s", zorder=3)
        ax.annotate(c.id, (c.lon, c.lat), textcoords="offset points", xytext=(8,-4), fontsize=9)
    for site in problem.sites:
        on = site.id in built
        ax.scatter(site.lon, site.lat, s=260 if on else 90, c="#2f855a" if on else "#a0aec0",
                   marker="^", zorder=4)
        ax.annotate(site.id, (site.lon, site.lat), textcoords="offset points", xytext=(8,4),
                    fontsize=9, fontweight="bold" if on else "normal",
                    color="#22543d" if on else "#718096")
    pos = {n.id:(n.lon,n.lat) for n in list(problem.sites)+list(problem.customers)}
    tlast = problem.periods[-1]
    for (i,j,t),q in s.shipments.items():
        if t==tlast and q>0:
            ax.plot([pos[i][0],pos[j][0]],[pos[i][1],pos[j][1]],"-",lw=max(1,q/2e5),
                    c="#2f855a", alpha=.6, zorder=2)
            ax.annotate(f"{q/1e3:.0f} kt", ((pos[i][0]+pos[j][0])/2,(pos[i][1]+pos[j][1])/2),
                        fontsize=8, color="#22543d")
    ax.set_title(title + f"  |  final-period flows"); ax.set_xlabel("lon"); ax.set_ylabel("lat")
    ax.set_aspect("equal"); plt.tight_layout(); plt.show()
    return s

base_solution = show(base, "Base case: Asahi-inspired east-coast DC network")
print()
print("Compare with Asahi's actual announced program: Deer Park (Melbourne west),")
print("Redbank (Ipswich, QLD) and a third Sydney DC. Green triangles = model's choice.")

In [ ]:
#@title 4. Try changing the world — move the sliders, then press ▶ on this cell { display-mode: "form" }
nsw_inbound_cost = 74  #@param {type:"slider", min:50, max:110, step:1}
vic_inbound_cost = 57  #@param {type:"slider", min:45, max:95, step:1}
demand_growth_extra_pct = 0  #@param {type:"slider", min:-20, max:60, step:5}
dc_capex_musd_per_250kt = 50  #@param {type:"slider", min:30, max:150, step:5}

trial = copy.deepcopy(base)
for s_ in trial.sites:
    if s_.id.endswith("_NSW"): s_.production_cost = nsw_inbound_cost + (2 if "Moorebank" in s_.id else 0)
    if s_.id.endswith("_VIC"): s_.production_cost = vic_inbound_cost + (1 if "Truganina" in s_.id else 0)
    s_.expansion_capex = dc_capex_musd_per_250kt * 1e6 * (s_.expansion_capex/5e7)
for c in trial.customers:
    c.demand = {t: v*(1+demand_growth_extra_pct/100) for t,v in c.demand.items()}

dq2 = run_checks(trial)
print(dq2.summary())
try:
    s2 = show(trial, "Your scenario")
    d = (s2.objective - base_solution.objective)/1e6
    print(f"\nDifference vs base case: {d:+,.0f} MUSD NPV")
except Exception as e:
    print("The model refused, with a diagnosis (this is a feature):")
    print(" ", e)

### What to notice
- **Push `nsw_inbound_cost` above ~95** and watch NSW's own DC disappear — Sydney gets served
  from interstate instead. Somewhere in between the answer is a **near-tie**, and a tie is the
  honest answer, not a decision.
- **Drop `dc_capex` low** and the model buys capacity everywhere; push it high and it
  concentrates in fewer, larger sites.
- **Push demand growth very high** and eventually the model *refuses with a diagnosis*
  (site maxima can't cover demand). A trustworthy optimiser says *why* it can't answer.
- Every run is deterministic (single thread, fixed seed) and carries a reproducibility record —
  the same inputs give the same answer on any machine.

The controlled version of this — golden problems, adversarial audits, change control, the five
agent roles — lives in the [repository](https://github.com/jkeetley1/manufacturing).